# 08 · Índice común: definición, datos y protocolo de validación
Cambio de objetivo aprobado por el usuario: una escala común, sin fijar Aqua
como verdad de terreno. El notebook 07 permanece como baseline comparativo.
Este notebook no ajusta el índice ni abre la prueba final 2024–2025.
No se considera que un índice sea correcto por hacer coincidir las curvas.

## 1. Qué queremos medir
Actividad térmica detectable relativa por zona y semana, con efecto de sensor
y oportunidades de observación modelados explícitamente. No número real de
incendios, probabilidad de ignición ni superficie quemada.
Sea A(g,t) la actividad común estimada y B(g) su media en un período base de
desarrollo fijo. Definimos I(g,t)=100*A(g,t)/B(g), solo si B(g)>0 y hay soporte
suficiente. I=100 es el promedio histórico de esa zona; I=200 significa el doble
de actividad estimada respecto a esa base, no el doble de incendios.
No usar mínimo/máximo de toda la historia ni añadir un epsilon oculto si B=0.
Índices con bases locales comparan desviaciones relativas, no magnitudes
absolutas entre regiones. Se conserva también A y su unidad/soporte definidos.

## 2. Modelo candidato, no garantía matemática
Propuesta a contrastar: modelo conjunto de observación, con conteos Poisson
como baseline y binomial negativa si la dispersión observada lo justifica.
Una forma inicial es log(mu_sgt)=log(E_sgt)+eta_gt+alpha_s+delta_s,estacion.
E representa área-tiempo de observación válida en un soporte geográfico común;
no es simplemente número de puntos ni número bruto de píxeles 375m frente a 1km.
eta es actividad latente común y alpha/delta son diferencias de respuesta.
La detección depende también de intensidad/tamaño: probar extensiones solo
si mejoran validación y conservan identificabilidad y estabilidad.
Se requieren restricciones de identificación (por ejemplo efectos de sensor
con suma cero) y conectividad de los períodos de solapamiento. No se pueden
estimar libremente actividad, cobertura y sensibilidad a partir de positivos solos.
Fijar la escala no convierte la actividad latente en una verdad física observada.

In [1]:
from pathlib import Path
import os,json
import pandas as pd
from IPython.display import display
ROOT=Path.cwd().resolve()
if ROOT.name in {'notebooks','validacion'}:
    ROOT=ROOT.parent
ARTIFACTS=Path(os.environ.get('FIRMS_ARTIFACTS_DIR',str(ROOT/'artifacts')))
PREP=ARTIFACTS/'02_preparacion'
OUT=ARTIFACTS/'09_indice_comun'
OUT.mkdir(parents=True,exist_ok=True)
inventory=pd.read_csv(PREP/'inventario.csv')
display(inventory[['source_member','processing','rows','start','end']])

,source_member,processing,rows,start,end
0,fire_archive_M-C61_848898.csv,SP,362159,2000-11-01 04:16:00+00:00,2026-08-31 20:44:00+00:00
1,fire_nrt_M-C61_848898.csv,NRT,4264,2026-09-01 02:04:00+00:00,2026-11-13 02:02:00+00:00
2,fire_archive_J1V-C2_848899.csv,SP,641945,2018-04-01 05:13:00+00:00,2026-07-31 18:13:00+00:00
3,fire_nrt_J1V-C2_848899.csv,NRT,44689,2026-08-01 05:18:00+00:00,2026-11-13 18:47:00+00:00
4,fire_archive_J2V-C2_848900.csv,SP,243041,2023-03-01 06:21:00+00:00,2026-07-31 18:59:00+00:00
5,fire_nrt_J2V-C2_848900.csv,NRT,41185,2026-08-01 06:01:00+00:00,2026-11-14 06:33:00+00:00
6,fire_archive_SV-C2_848901.csv,SP,1058423,2012-01-20 06:33:00+00:00,2026-07-31 19:34:00+00:00
7,fire_nrt_SV-C2_848901.csv,NRT,43565,2026-08-01 06:37:00+00:00,2026-11-01 18:51:00+00:00


## 3. Auditoría de requisitos
Los campos exigidos en esta tabla son un contrato interno de la futura tabla
de exposición; no se presume que NASA utilice estos mismos nombres de columna.
Se examina presencia de esquema, no se valida todavía su semántica o calidad.
Máscaras de fuego/observación completas pueden distinguir cloud, missing,
non-fire y fire. Los CSV de detecciones no proporcionan todos esos estados.
Fuente NASA: https://appliedsciences.nasa.gov/sites/default/files/2025-05/Part%203%20FIRMS%202025%20Q%26A%20Session.docx.pdf
Algoritmo VIIRS: https://viirsland.gsfc.nasa.gov/PDF/VIIRS_activefire_375m_ATBD.pdf

In [2]:
required_exposure={'valid_observed_area_km2','observation_time_utc','platform','cell_id','fire_mask_class'}
checks=[]
for _,row in inventory.iterrows():
    path=PREP/row.output_file
    columns=set(pd.read_csv(path,nrows=0).columns)
    missing=sorted(required_exposure-columns)
    checks.append({'file':row.source_member,'exposure_schema_present':not missing,
                   'missing_internal_fields':','.join(missing)})
checks=pd.DataFrame(checks)
display(checks)
exposure_ready=bool(checks.exposure_schema_present.all())
assert exposure_ready is False, 'Si se agregan fuentes de exposición, revisar contrato y semántica antes de entrenar.'
checks.to_csv(OUT/'auditoria_exposicion.csv',index=False)
print('Fuentes positivas disponibles; exposición observacional completa todavía no incorporada.')

,file,exposure_schema_present,missing_internal_fields
0,fire_archive_M-C61_848898.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
1,fire_nrt_M-C61_848898.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
2,fire_archive_J1V-C2_848899.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
3,fire_nrt_J1V-C2_848899.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
4,fire_archive_J2V-C2_848900.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
5,fire_nrt_J2V-C2_848900.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
6,fire_archive_SV-C2_848901.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
7,fire_nrt_SV-C2_848901.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."


Fuentes positivas disponibles; exposición observacional completa todavía no incorporada.


## 4. Pipeline de validación
Se ejecutan capas complementarias; ninguna sustituye a las demás.
La precisión/recall de incendios exige referencias representativas y casos
negativos verificables. Los cuatro reportes positivos de INDECI no bastan.

In [3]:
pipeline=pd.DataFrame([
 {'stage':'Datos','checks':'Esquema, UTC, coordenadas, versiones, duplicados y trazabilidad','reuse':'01_preparacion_firms','status':'base implementada'},
 {'stage':'Observación','checks':'Pasos, máscara, nubes, área válida, ceros observados frente a faltantes','reuse':'fuentes nuevas de máscaras/exposición','status':'pendiente de datos'},
 {'stage':'Espacial','checks':'R-tree más distancia geodésica; celdas coherentes; bloques espaciales reservados y separación entre bloques','reuse':'02_organizacion_espacio_temporal y 05_contraste_curimana','status':'asignación implementada; validación espacial externa pendiente'},
 {'stage':'Temporal','checks':'Backtesting por bloques; normalización y selección ajustadas dentro de cada fold; prueba final intacta','reuse':'03_diagnostico y 07_baseline','status':'baseline disponible; protocolo común pendiente'},
 {'stage':'Entre plataformas','checks':'Aqua/Terra/SNPP/NOAA20/NOAA21 separados; dejar una plataforma fuera; concordancia y discontinuidad al cambiar','reuse':'períodos de solapamiento y datos ya descargados','status':'pendiente'},
 {'stage':'Evidencia externa','checks':'Fecha del hecho, georreferencia e incertidumbre; no atribuir candidatos automáticamente','reuse':'04_auditoria_json y validacion/05_contraste_curimana','status':'caso exploratorio implementado'},
 {'stage':'Revisión humana','checks':'Coherencia geográfica, fecha de imagen, notas y exportación','reuse':'validacion/06_google_maps_reutilizable','status':'integración pendiente de API key real'},
 {'stage':'Robustez','checks':'Región/estación/día-noche/resolución; sesgo residual por sensor; incertidumbre y faltantes','reuse':'07 como control, no objetivo final','status':'pendiente para índice común'}
])
display(pipeline)
pipeline.to_csv(OUT/'pipeline_validacion.csv',index=False)

,stage,checks,reuse,status
0,Datos,"Esquema, UTC, coordenadas, versiones, duplicad...",01_preparacion_firms,base implementada
1,Observación,"Pasos, máscara, nubes, área válida, ceros obse...",fuentes nuevas de máscaras/exposición,pendiente de datos
2,Espacial,R-tree más distancia geodésica; celdas coheren...,02_organizacion_espacio_temporal y 05_contrast...,asignación implementada; validación espacial e...
3,Temporal,Backtesting por bloques; normalización y selec...,03_diagnostico y 07_baseline,baseline disponible; protocolo común pendiente
4,Entre plataformas,Aqua/Terra/SNPP/NOAA20/NOAA21 separados; dejar...,períodos de solapamiento y datos ya descargados,pendiente
5,Evidencia externa,"Fecha del hecho, georreferencia e incertidumbr...",04_auditoria_json y validacion/05_contraste_cu...,caso exploratorio implementado
6,Revisión humana,"Coherencia geográfica, fecha de imagen, notas ...",validacion/06_google_maps_reutilizable,integración pendiente de API key real
7,Robustez,Región/estación/día-noche/resolución; sesgo re...,"07 como control, no objetivo final",pendiente para índice común


## 5. Evaluación que debe fijarse antes del ajuste común
- Reservar 2024–2025 para prueba final; no se ha evaluado ese bloque.
- Revisar desarrollo/calibración para NOAA-21: empieza en 2023 y por eso no
  puede aprenderse su efecto usando únicamente 2013–2021.
- Definir bloques espaciales y buffers antes de mirar sus errores. No mezclar
  celdas vecinas del mismo episodio entre ajuste y evaluación independiente.
- Estimar exposición, normalización, efectos de sensor e hiperparámetros solo
  dentro del bloque de ajuste de cada fold temporal/espacial.
- Evaluar observaciones retenidas: MAE, sesgo, deviance apropiada al modelo,
  cobertura de intervalos y estabilidad por segmentos; no medir error contra
  el propio índice latente como si fuese verdad de terreno.
- Prueba de transición: retirar un sensor/plataforma y comprobar continuidad
  usando los restantes, sin forzar que desaparezcan cambios ambientales reales.
- Congelar el método antes de abrir la prueba final. Si se modifica después,
  esa prueba deja de ser independiente y debe declararse exploratoria.

In [4]:
contract={
 'objective':'common_relative_detectable_thermal_activity_index',
 'reference_sensor':None,'physical_truth_claim':False,'index_fitted':False,
 'baseline_notebook_07_retained':True,'base_definition':'100 times estimated common activity / fixed development mean for each region',
 'base_zero_policy':'not estimable; no hidden epsilon',
 'coverage_adjusted_model_ready':exposure_ready,
 'needs':['observational_exposure_and_fire_masks','NOAA20_NOAA21_overlap_analysis','identified_joint_observation_model','predeclared_spatial_and_temporal_validation'],
 'test_2024_2025_evaluated':False,
 'four_positive_reports_establish_global_precision_recall':False,
 'google_maps_confirms_2019_fire_automatically':False,
 'next_step':'audit availability of observation-mask products and select one pilot region/time window',
}
(OUT/'contrato_indice_comun.json').write_text(json.dumps(contract,ensure_ascii=False,indent=2),encoding='utf-8')
print('Contrato registrado. No se entrenó un índice común con exposición desconocida.')

Contrato registrado. No se entrenó un índice común con exposición desconocida.
